# 02 — PySpark Pipeline Walkthrough

This notebook walks through the generated PySpark ETL pipeline in detail:
- How the Jinja2 templates produce production-ready PySpark code
- SparkSession configuration with Delta Lake
- All supported transformation operations
- Delta Lake CRUD operations (Create, Read, Update, Delete, Merge)
- Query optimization (AQE, broadcast hints, partition tuning)

> ⚙️ **Prerequisite**: Java 17 must be installed for PySpark to run.

## 1. ⚙️ Setup

In [ ]:
import sys
sys.path.insert(0, '../../src')

import warnings
warnings.filterwarnings('ignore')

print('Python:', sys.version.split()[0])

## 2. SparkSession with Delta Lake

The agent always creates a SparkSession with the mandatory Delta Lake extensions:

In [ ]:
from etl_agent.spark.session import get_or_create_spark

# Creates/retrieves a SparkSession with Delta Lake configured
spark = get_or_create_spark(app_name='ETL-Walkthrough-Demo')

print(f'Spark version  : {spark.version}')
print(f'Delta extension: {spark.conf.get("spark.sql.extensions", "not set")}')
print(f'Delta catalog  : {spark.conf.get("spark.sql.catalog.spark_catalog", "not set")}')

## 3. Apply All Optimizations

In [ ]:
from etl_agent.spark.optimizer import apply_all_optimizations

# Apply AQE, dynamic partition pruning, broadcast threshold
apply_all_optimizations(spark, broadcast_threshold_mb=10)

print('AQE enabled     :', spark.conf.get('spark.sql.adaptive.enabled'))
print('AQE coalesce    :', spark.conf.get('spark.sql.adaptive.coalescePartitions.enabled'))
print('AQE skew join   :', spark.conf.get('spark.sql.adaptive.skewJoin.enabled'))
print('DPP enabled     :', spark.conf.get('spark.sql.optimizer.dynamicPartitionPruning.enabled'))
print('Broadcast bytes :', spark.conf.get('spark.sql.autoBroadcastJoinThreshold'))

## 4. Sample Data

In [ ]:
from datetime import date, timedelta
from pyspark.sql.types import *
from pyspark.sql import functions as F

today = date.today()

orders_data = [
    ('ORD-001', 'CUST-A', today - timedelta(5),   120.0, 2, 'Electronics'),
    ('ORD-002', 'CUST-A', today - timedelta(10),   80.0, 1, 'Books'),
    ('ORD-003', 'CUST-B', today - timedelta(90),  200.0, 3, 'Electronics'),
    ('ORD-004', 'CUST-C', today - timedelta(200),  50.0, 1, 'Clothing'),
    ('ORD-005', 'CUST-D', today - timedelta(365), 500.0, 5, 'Electronics'),
    ('ORD-006', None,     today - timedelta(1),    30.0, 1, 'Books'),  # null customer
]

schema = StructType([
    StructField('order_id',    StringType(),  False),
    StructField('customer_id', StringType(),  True),
    StructField('order_date',  DateType(),    True),
    StructField('total',       DoubleType(),  True),
    StructField('qty',         IntegerType(), True),
    StructField('category',    StringType(),  True),
])

orders_df = spark.createDataFrame(orders_data, schema=schema)
print(f'Orders DataFrame: {orders_df.count()} rows')
orders_df.show(truncate=False)

## 5. Transformation Operations

The agent generates code for 9 transformation types. Let's run each one:

In [ ]:
# 1. FILTER
df_filtered = orders_df.filter(F.col('customer_id').isNotNull())
print(f'filter (remove nulls): {df_filtered.count()} rows (was {orders_df.count()})')

In [ ]:
# 2. FILL_NULL
df_filled = orders_df.fillna({'customer_id': 'UNKNOWN', 'qty': 1})
print('fill_null: nulls replaced')
df_filled.filter(F.col('customer_id') == 'UNKNOWN').show()

In [ ]:
# 3. RENAME
df_renamed = orders_df.withColumnRenamed('total', 'order_total') \
                      .withColumnRenamed('qty', 'quantity')
print('rename: columns renamed to', df_renamed.columns)

In [ ]:
# 4. CAST
df_cast = orders_df.withColumn('total', F.col('total').cast('decimal(10,2)'))
df_cast.printSchema()

In [ ]:
# 5. AGGREGATE
df_agg = orders_df.groupBy('customer_id').agg(
    F.count('order_id').alias('order_count'),
    F.sum('total').alias('total_spent'),
    F.max('order_date').alias('last_order'),
)
print('aggregate: customer summaries')
df_agg.show()

In [ ]:
# 6. ENRICH (derived columns)
df_enriched = orders_df.withColumn(
    'recency_days', F.datediff(F.current_date(), F.col('order_date'))
).withColumn(
    'is_recent', F.col('recency_days') <= 30
).withColumn(
    'revenue_per_item', F.col('total') / F.col('qty')
)
print('enrich: derived columns added')
df_enriched.select('order_id', 'recency_days', 'is_recent', 'revenue_per_item').show()

In [ ]:
# 7. DEDUPE
dupes = orders_data + [('ORD-001', 'CUST-A', today - timedelta(5), 120.0, 2, 'Electronics')]  # duplicate
df_with_dupes = spark.createDataFrame(dupes, schema=schema)
df_deduped = df_with_dupes.dropDuplicates(['order_id', 'customer_id'])
print(f'dedupe: {df_with_dupes.count()} → {df_deduped.count()} rows')

In [ ]:
# 8. SORT
df_sorted = orders_df.orderBy(F.col('total').desc(), F.col('order_date').desc())
print('sort: by total desc, order_date desc')
df_sorted.show(3)

## 6. Delta Lake CRUD Operations

In [ ]:
import tempfile
from etl_agent.spark.templates.delta_ops_generated import (
    write_delta, read_delta, update_delta, delete_delta, merge_delta, get_delta_history
)

# Use a temp directory for demo
tmp_dir = tempfile.mkdtemp()
delta_path = f'{tmp_dir}/orders_delta'

# CREATE — write initial data
write_delta(orders_df.filter(F.col('customer_id').isNotNull()), delta_path)
print(f'✅ Created Delta table at {delta_path}')

# READ
df_read = read_delta(spark, delta_path)
print(f'✅ Read {df_read.count()} rows from Delta table')

In [ ]:
# UPDATE — mark Electronics as high-value
update_delta(
    spark, delta_path,
    condition="category = 'Electronics'",
    update_map={'total': 'total * 1.1'},  # 10% markup
)
print('✅ Updated Electronics orders (+10% total)')
read_delta(spark, delta_path).filter(F.col('category') == 'Electronics').show()

In [ ]:
# MERGE (upsert) — add new orders
new_orders = spark.createDataFrame([
    ('ORD-001', 'CUST-A', today - timedelta(5), 135.0, 2, 'Electronics'),  # update existing
    ('ORD-007', 'CUST-E', today,                 75.0, 1, 'Books'),          # new insert
], schema=schema)

result = merge_delta(
    spark, new_orders, delta_path,
    merge_keys=['order_id'],
    insert_all=True,
)
print('✅ Merge completed:', result)
read_delta(spark, delta_path).show()

In [ ]:
# DELETE — remove old orders (>300 days)
delete_delta(spark, delta_path, condition='recency_days > 300')
# Note: recency_days doesn't exist in this table — use total as a proxy
delete_delta(spark, delta_path, condition="customer_id = 'CUST-D'")
print(f'✅ Deleted old records. Remaining: {read_delta(spark, delta_path).count()}')

In [ ]:
# Delta time-travel — read previous version
df_v0 = read_delta(spark, delta_path, version=0)
print(f'✅ Time-travel to version 0: {df_v0.count()} rows')

# Delta history
history = get_delta_history(spark, delta_path)
print('✅ Operation history:')
history.select('version', 'operation', 'operationMetrics').show(truncate=50)

## 7. Spark Optimizer Demo

In [ ]:
from etl_agent.spark.optimizer import broadcast_join, repartition_for_write, coalesce_small_output

# Create a small lookup table (will be broadcast)
categories = spark.createDataFrame([
    ('Electronics', 'High Margin'),
    ('Books', 'Low Margin'),
    ('Clothing', 'Medium Margin'),
], ['category', 'margin_type'])

# Broadcast join — categories table is tiny, auto-broadcast
df_joined = broadcast_join(orders_df, categories, join_keys='category', join_type='left')
print('Broadcast join result:')
df_joined.select('order_id', 'category', 'margin_type').show(truncate=False)

# Coalesce small output
df_coalesced = coalesce_small_output(df_joined, max_partitions=2)
print(f'Partitions after coalesce: {df_coalesced.rdd.getNumPartitions()}')

## 8. Jinja2 Template Preview

Let's see how the Jinja2 template renders for the RFM analysis story:

In [ ]:
import yaml
from jinja2 import Environment, FileSystemLoader
from datetime import datetime

template_dir = '../../src/etl_agent/spark/templates'
env = Environment(loader=FileSystemLoader(template_dir))

with open('../../config/story_examples/rfm_analysis.yaml') as f:
    story = yaml.safe_load(f)

# Render the README template
template = env.get_template('pipeline_readme.md.j2')
rendered = template.render(
    pipeline_name=story['id'],
    story_id=story['id'],
    description=story.get('description', ''),
    generated_at=datetime.now().isoformat(),
    pipeline_version='1.0.0',
    source=story['source'],
    target=story['target'],
    transformations=story.get('transformations', []),
    acceptance_criteria=story.get('acceptance_criteria', []),
    tags=story.get('tags', []),
)

print(rendered[:1200] + '\n... (truncated)')

In [ ]:
# Cleanup
spark.stop()
print('SparkSession stopped')